# 00. Setup, assumptions and data audit

**Purpose.** Everything every other notebook relies on: the shared assumptions, the units, and a quality check of all eight data files. Run this first.

**How the notebooks connect.** Each module notebook ends by saving one small file in `results/`. Notebook 10 reads them all and builds the ₹1,000 crore allocation.

| Step | Notebook | Saves |
|---|---|---|
| 1 | 01 Strategic initiatives | `results/initiatives.json` |
| 2 | 02 Credit risk | `results/credit.json` |
| 3 | 03 Fraud | `results/fraud.json` |
| 4 | 04 Churn and CLV | `results/churn.json` |
| 5 | 05 Equity returns | `results/equity_predictions.csv` |
| 6 | 06 Portfolio | `results/portfolio.json` |
| 7 | 07 HFT | `results/hft.json` |
| 8 | 08 Options | `results/options.json` |
| 9 | 09 Liquidity | `results/liquidity.json` |
| 10 | 10 Allocation | final tables |

In [1]:
import sys; sys.path.insert(0, '../src')
from novafin import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

NovaFin helpers loaded. Root: /mnt/user-data/outputs/novafin_capstone


## 1. Shared assumptions
These are used by every module. Change them here (in `src/novafin.py`) and re-run the notebooks to see the effect everywhere.

In [2]:
pd.DataFrame({'Value': ASSUMPTIONS}).rename_axis('Assumption')

,Value
Assumption,
LGD,0.40
DISCOUNT_RATE,0.12
RISK_FREE,0.07
VAR_CONFIDENCE,0.99
COST_OF_FUNDS,0.06
MAX_ECL_RATIO,0.03
FRAUD_MISS_COST,10000.00
FRAUD_FALSE_POSITIVE_COST,500.00
TOTAL_CAPITAL_CR,1000.00


**Why these values.**
- **LGD 40%, fraud costs ₹10,000 / ₹500, ₹1,000 crore, 1,000 contacts:** given in the brief.
- **Discount rate 12%:** a typical hurdle rate for corporate projects in India. Used for NPV in notebook 01.
- **Risk-free 7%:** roughly the Indian 10-year government bond yield. Used for Sharpe ratios.
- **Cost of funds 6%:** funding cost for lending margins (notebook 02).
- **Max ECL ratio 3%:** the risk appetite for the loan book: expected credit loss should not exceed 3% of approved exposure.
- **VaR 99%:** a standard regulatory-style confidence level.

These are assumptions, not facts from the data, and the report should say so.

## 2. Data inventory and quality checks

In [3]:
names = ['initiatives','loans','transactions','customers','market','hft','options','liquidity']
rows = []
data = {}
for n in names:
    d = load(n); data[n] = d
    rows.append({'dataset': n, 'rows': len(d), 'columns': d.shape[1],
                 'missing cells': int(d.isna().sum().sum()),
                 'duplicate rows': int(d.duplicated().sum()),
                 'numeric cols': int(d.select_dtypes('number').shape[1])})
inventory = pd.DataFrame(rows).set_index('dataset')
inventory

,rows,columns,missing cells,duplicate rows,numeric cols
dataset,,,,,
initiatives,180,14,0,0,12
loans,5000,14,0,0,13
transactions,30000,12,0,0,8
customers,5000,13,0,0,13
market,21855,15,0,0,12
hft,120000,33,0,0,27
options,10000,11,0,0,8
liquidity,1216,13,0,0,12


No missing values and no duplicate rows anywhere, so cleaning is about ranges and types, not imputation. Next: target variables and class balance.

In [4]:
targets = {'initiatives': 'Historical_Success', 'loans': 'Default_Flag', 'transactions': 'Fraud_Flag',
           'customers': 'Churn_Flag', 'hft': 'Price_Move_Class'}
out = []
for n, t in targets.items():
    vc = data[n][t].value_counts(normalize=True).round(3)
    out.append({'dataset': n, 'target': t, 'class balance': dict(vc)})
pd.DataFrame(out).set_index('dataset')

,target,class balance
dataset,,
initiatives,Historical_Success,"{1: 0.767, 0: 0.233}"
loans,Default_Flag,"{0: 0.801, 1: 0.199}"
transactions,Fraud_Flag,"{0: 0.977, 1: 0.023}"
customers,Churn_Flag,"{0: 0.982, 1: 0.018}"
hft,Price_Move_Class,"{'DOWN': 0.419, 'UP': 0.417, 'FLAT': 0.164}"


**Reading this.** Fraud (2.3%) and churn (1.8%) are rare, which is why accuracy is meaningless there and PR-AUC, recall and cost-based thresholds are needed. Defaults (20%) are moderately imbalanced. HFT classes are fairly balanced, with FLAT (16%) the smallest. Initiatives succeed 77% of the time.

## 3. Units: are the money columns rupees or lakh?

In [5]:
loans, init = data['loans'], data['initiatives']
print('Average annual income in loans file:', f"{loans.Annual_Income.mean():,.0f}")
print('Average loan amount:               ', f"{loans.Loan_Amount.mean():,.0f}")
print('Total loan book (rupees crore):    ', f"{cr(loans.Loan_Amount.sum()):,.0f}")
print()
print(init[['Initial_Investment','Revenue_Year1','Operating_Cost']].describe().round(1).loc[['mean','min','max']])

Average annual income in loans file: 1,051,769
Average loan amount:                1,279,306
Total loan book (rupees crore):     640

      Initial_Investment  Revenue_Year1  Operating_Cost
mean               157.5           56.7            35.9
min                 31.5            9.5             5.2
max                296.5          137.8            95.7


**Decision.** The README says money is in INR lakh unless the column context says otherwise. Read as lakh, the average borrower would earn about ₹1,000 billion a year, which is absurd, so the loan and customer files are in **rupees** (average income about ₹10.5 lakh). The strategic initiatives file has much smaller numbers (investments between about 31 and 297) and is plausibly in **₹ crore**, which fits a ₹1,000 crore budget. Notebook 01 treats it as crore; the Fund/Review/Reject decision depends only on ratios, so it does not change if the unit is different.

The loan book in the file is about ₹640 crore, which is a useful scale check against the ₹1,000 crore to allocate.

## 4. Time structure: which files need time-based splits?

In [6]:
mk, tx, hft, liq, opt = data['market'], data['transactions'], data['hft'], data['liquidity'], data['options']
print('market   : ', mk.Date.min(), 'to', mk.Date.max(), '|', mk.Ticker.nunique(), 'tickers')
print('transactions:', tx.Timestamp.min(), 'to', tx.Timestamp.max())
print('hft      : ', hft.Trading_Day.nunique(), 'trading days,', hft.Stock.nunique(), 'stocks; timestamp unique values:', hft.Timestamp.nunique(), 'of', len(hft), 'rows')
print('liquidity:', liq.Date.min(), 'to', liq.Date.max())
print('options  : expiry range', opt.Expiry.min(), 'to', opt.Expiry.max())

market   :  2021-01-29 to 2026-08-31 | 15 tickers
transactions: 2025-01-01 00:00:00.000000000 to 2026-08-31 00:00:00.000000000
hft      :  20 trading days, 10 stocks; timestamp unique values: 7506 of 120000 rows
liquidity: 2022-01-03 to 2026-08-31
options  : expiry range 2026-09-13 to 2028-08-31


**Consequences for validation**
- Market, liquidity and transactions: split by **time**, never randomly.
- HFT: timestamps only go to the minute, so rows within a minute cannot be ordered by time. Split by **trading day** and keep the file order.
- Loans, customers, initiatives: stratified random splits are fine (cross-sectional data).
- Options: random split is acceptable for a cross-section, but a split by underlying or expiry is a tougher and fairer test; notebook 08 discusses this.

## 5. Leakage watch-list (decided now, applied in each notebook)

In [7]:
pd.DataFrame([
 ['hft', 'Future_Return_100ms, Future_Price_100ms', 'They are the answer to Price_Move_Class'],
 ['customers', 'Estimated_CLV', 'Derived from customer value and retention, so it is related to churn. Keep out of churn features'],
 ['options', 'Black_Scholes_Price', 'Do not feed it to the model when comparing ML against Black-Scholes'],
 ['market', 'Return (same-day)', 'Target must be the next-period return, built by shifting'],
 ['loans', 'Interest_Rate', 'Known at origination so allowed, but it already embeds the lender\'s own risk view. Notebook 02 tests the model with and without it'],
], columns=['dataset','column(s)','issue']).set_index('dataset')

,column(s),issue
dataset,,
hft,"Future_Return_100ms, Future_Price_100ms",They are the answer to Price_Move_Class
customers,Estimated_CLV,"Derived from customer value and retention, so ..."
options,Black_Scholes_Price,Do not feed it to the model when comparing ML ...
market,Return (same-day),"Target must be the next-period return, built b..."
loans,Interest_Rate,"Known at origination so allowed, but it alread..."


## 6. What the brief and the files disagree on
- Loans have no `Loan_ID` column (only `Customer_ID`), so the notebooks use the row index as the loan ID.
- The HFT class is called **FLAT**, not No Move, and the target is `Future_Return_100ms`.
- `nova_liquidity.csv` is not listed in the README, and the brief has no module for it, but liquidity forecasting and stress testing are listed as skills. Notebook 09 covers it.
- Module numbers jump (1–6, 9, 10, 11). The notebooks are numbered 01–10 in the order of work, and module numbers are given in each title.

## Status
Setup is complete when this notebook runs top to bottom without errors. Next: **01 Strategic initiatives** or **02 Credit risk**.